# TimesFM-2.5 Benchmarking

| | |
|---|---|
| **Organization** | Google Research |
| **Version** | TimesFM-2.5 (September 2025) |
| **Parameters** | 200M |
| **Input Features** | Target series only (covariates available via external XReg wrapper, but requires JAX) |


## Key Characteristics

TimesFM follows a decoder-only transformer design in which non-overlapping patches of time series data are treated as tokens and processed autoregressively. The model was pretrained on approximately 100 billion real-world time series observations, including Google Trends and Wikipedia pageview data, supplemented by synthetic and augmented series.

Version 2.5 reduces the parameter count from 500M (v2.0) to 200M while extending the maximum context length from 2,048 to 16,384 time steps and introducing natively calibrated quantile outputs. At time of release, TimesFM-2.5 ranked first among zero-shot foundation models on the GIFT-Eval leaderboard by both MASE and CRPS.



---

**Validation split:** Train on periods 1–36, validate on 37–42 (identical to Notebook 6)  


**Target:** `Revenue cons. (anon)` at subsegment level  


**Runtime:** T4 GPU (Google Colab) — set manually: Runtime → Change runtime type → T4 GPU


## 1. Setup & Data Loading

In [1]:
!pip install -q --upgrade transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 104.3 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 637.4/637.4 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 136.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
granite-tsfm 0.3.2.dev37+g640f7c80a requires transformers[torch]<4.51,>=4.38.0, but you have transformers 5.5.0 which is incompatible.


In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')

GPU available: True
Device: Tesla T4


In [14]:
# ── Constants ──
PERIOD_COL = 'Anon Period'
TARGET     = 'Revenue cons. (anon)'
SUBSEG_COL = 'TGL Business Subsegment'
VAL_CUTOFF = 36
HORIZON    = 6

# ── Paths ──
from google.colab import drive
drive.mount('/content/drive')               
                                          
DATA_PATH = '/content/drive/MyDrive/Coding/data/features/training_subsegment_fs.parquet'      

# ── Load ──
full_train = pd.read_parquet(DATA_PATH)
train_df   = full_train[full_train[PERIOD_COL] <= VAL_CUTOFF].copy()
val_df     = full_train[full_train[PERIOD_COL] >  VAL_CUTOFF].copy()

print(f'Full train: {full_train.shape}')
print(f'Train (1–36): {train_df.shape} | Val (37–42): {val_df.shape}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Full train: (4237, 110)
Train (1–36): (3524, 110) | Val (37–42): (713, 110)


In [15]:
#   TimesFM does not support covariates — only the target series is used as input. 
#   Therefore Feature columns are not loaded.

# ── Subsegments ──
subsegments = sorted(
    train_df.dropna(subset=[TARGET])
    .groupby(SUBSEG_COL, observed=True)
    .size()
    .pipe(lambda s: s[s >= 2].index)
    .tolist()
)
print(f'Subsegments: {len(subsegments)}')

Subsegments: 117


In [ ]:
# ── Evaluation function ──
def evaluate(results: dict, label: str) -> dict:
    all_true, all_pred = [], []
    for seg, preds in results.items():
        actuals = (val_df[val_df[SUBSEG_COL] == seg]
                   .sort_values(PERIOD_COL)
                   .dropna(subset=[TARGET])[TARGET].values)
        n = min(len(actuals), len(preds))
        all_true.extend(actuals[:n])
        all_pred.extend(preds[:n])
    y, yh = np.array(all_true), np.array(all_pred)
    rmse  = np.sqrt(mean_squared_error(y, yh))
    wmape = np.sum(np.abs(y - yh)) / np.sum(np.abs(y)) * 100
    r2    = r2_score(y, yh)
    print(f'{label} — RMSE: {rmse:,.0f} | wMAPE: {wmape:.1f}% | R²: {r2:.4f}')
    return {'Model': label, 'RMSE': rmse, 'wMAPE': wmape, 'R2': r2}

# ── Results tracker ──
all_metrics = []

---

200M-parameter decoder-only foundation model.  
16K context length.  
Zero-shot — no fine-tuning.

- Input must be padded to a multiple of `PATCH_SIZE=32`
- Output: `mean_predictions` of shape `(batch, horizon)`

In [6]:
from transformers import TimesFm2_5ModelForPrediction

device = 'cuda' if torch.cuda.is_available() else 'cpu'

tfm_model = TimesFm2_5ModelForPrediction.from_pretrained(
    'google/timesfm-2.5-200m-transformers',
)
tfm_model = tfm_model.to(torch.float32).to(device).eval()

print(f'TimesFM-2.5 loaded on {device}.')

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/925M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

TimesFM-2.5 loaded on cuda.


In [9]:
tfm_results, tfm_errors = {}, []

PATCH_SIZE = 32  # TimesFM internal patch size — input length must be a multiple

for i, seg in enumerate(subsegments):
    train_s = (train_df[train_df[SUBSEG_COL] == seg]
               .sort_values(PERIOD_COL)
               .dropna(subset=[TARGET]))
    val_s = (val_df[val_df[SUBSEG_COL] == seg]
             .sort_values(PERIOD_COL)
             .dropna(subset=[TARGET]))
    if len(train_s) < 4 or len(val_s) == 0:
        continue
    try:
        series = train_s[TARGET].values.astype(float)

        # Pad front to nearest multiple of PATCH_SIZE
        padded_len = ((len(series) + PATCH_SIZE - 1) // PATCH_SIZE) * PATCH_SIZE
        if padded_len > len(series):
            pad = np.full(padded_len - len(series), series[0])
            series = np.concatenate([pad, series])

        horizon = len(val_s)
        series_tensor = torch.tensor(series, dtype=torch.float32)

        with torch.no_grad():
            outputs = tfm_model(
                past_values=[series_tensor.to(device)],
                forecast_context_len=len(series),
            )
        preds = outputs.mean_predictions[0, :horizon].cpu().numpy()
        tfm_results[seg] = preds
    except Exception as e:
        tfm_errors.append((seg, str(e)))

    if (i + 1) % 25 == 0:
        print(f'{i+1}/{len(subsegments)}')

print(f'Done. Results: {len(tfm_results)} | Errors: {len(tfm_errors)}')
if tfm_errors:
    for seg, err in tfm_errors[:3]:
        print(f'  {seg}: {err}')

25/117
50/117
75/117
100/117
Done. Results: 107 | Errors: 0


In [10]:
metrics_tfm = evaluate(tfm_results, 'TimesFM-2.5')
all_metrics.append(metrics_tfm)

TimesFM-2.5 — RMSE: 9,182,434 | wMAPE: 11.4% | R²: 0.9819


In [ ]:
# ── Export
# rows = [{'Subsegment': seg, 'Period': t + 37, 'TimesFM_pred': p}
#         for seg, preds in tfm_results.items()
#         for t, p in enumerate(preds)]
# pd.DataFrame(rows).to_csv('/content/drive/MyDrive/Coding/data/predictions/timesfm_val_results.csv', index=False)
# print('Saved.')